In [1]:
import sys

In [2]:
sys.path.append('C:/Users/chris/PycharmProjects/data-quality-ml-research/data_quality_ml_research')

In [3]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

import helpers as dq
import mutation
import numpy as np
import pandas as pd

In [4]:
config = {
        "model": "Logistic Regression",
        "data_location": "../data/online_shoppers_intention.csv",
        "preprocess_data": False,
        "mutator": "change_to_zero",
        "num_scoring_groups": 5,
        "seeds": {"outer_seed": 123, "inner_seed": 456},
    }

In [5]:
clf = LogisticRegressionCV(Cs=[0.1, 1, 10], fit_intercept=True)

In [6]:
data = dq.read_data("C:/Users/chris/PycharmProjects/data-quality-ml-research/data/online_shoppers_intention.csv")

In [7]:
data.drop(["Month", "VisitorType", "Weekend"], axis=1, inplace=True)

In [8]:
feature_cols = data.columns.to_list()[:-1]
target_col = data.columns.to_list()[-1]

In [9]:
scoring_group_size = (1 - 0.5) / config['num_scoring_groups']

In [10]:
scoring_proportions = [0.5]
for i in range(config['num_scoring_groups']):
    scoring_proportions.append(scoring_group_size)

In [11]:
datasets = dq.random_nonintersecting_subsets(data, proportions=scoring_proportions, seed=config['seeds']['inner_seed'])

In [12]:
clf.fit(datasets[0][feature_cols], datasets[0][target_col])

C:\Users\chris\PycharmProjects\data-quality-ml-research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\chris\PycharmProjects\data-quality-ml-research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://sciki

LogisticRegressionCV(Cs=[0.1, 1, 10])

In [13]:
print(confusion_matrix(datasets[1][target_col], clf.predict(datasets[1][feature_cols])))

[[1028   24]
 [ 112   69]]


In [14]:
len(datasets)

6

In [15]:
datasets.pop(0)

,index,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,OperatingSystems,Browser,Region,TrafficType,Revenue
0,0,2,73.750,0,0.0,20,546.669697,0.000000,0.004394,0.000000,0.0,2,2,1,2,False
1,1,0,0.000,0,0.0,3,27.000000,0.066667,0.133333,0.000000,0.0,2,2,1,13,False
2,2,0,0.000,0,0.0,21,248.833333,0.010000,0.022500,0.000000,0.0,2,5,6,3,False
3,3,5,58.000,0,0.0,10,657.261905,0.000000,0.004167,0.000000,0.0,1,1,2,2,False
4,4,0,0.000,0,0.0,3,54.400000,0.000000,0.066667,0.000000,0.0,1,1,3,3,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6160,6160,10,163.425,0,0.0,73,1077.511667,0.000000,0.005833,0.000000,0.0,2,2,5,4,False
6161,6161,4,17.500,3,171.0,6,101.000000,0.020000,0.030000,0.000000,0.0,1,2,1,8,False
6162,6162,1,21.200,0,0.0,53,2224.633333,0.011538,0.035147,0.000000,0.0,2,2,3,3,False
6163,6163,1,0.000,0,0.0,16,1296.600000,0.022222,0.033333,13.283556,0.0,3,2,4,2,True


In [16]:
datasets[0]['PageValues'].mean()

5.790292049308191

In [17]:
mutator = mutation.Mutator()

In [18]:
def get_mutations(data, severity):
    mutator = mutation.Mutator()
    mutated_data = []
    for i in data:
        mutated = mutator.change_to_zero(i, columns=['PageValues']\
                , percentage=sev, seed=config['seeds']['outer_seed'], seed_modifier=config['seeds']['inner_seed'])
        mutated_data.append(mutated)   
    return mutated_data

In [19]:
mutated_data = []
for i in datasets:
    mutated = mutator.change_to_zero(i, columns=['PageValues']\
            , percentage=0.5, seed=config['seeds']['outer_seed'], seed_modifier=config['seeds']['inner_seed'])
    mutated_data.append(mutated)

In [20]:
mutated_data[0]['PageValues'].mean()

2.7873565491735603

In [21]:
datasets[0]['PageValues'].mean()

5.790292049308191

In [22]:
run_results = {}
run_results['mutation'] = 'change_to_zero'
run_results['severity'] = 0.5

In [23]:
classification_metrics = {
    "precision": precision_score,
    "recall": recall_score,
    "accuracy": accuracy_score,
    "f1": f1_score
}

In [24]:
def get_metrics(datasets, metric_dict):
    group_result = {}
    for metricName, metFunc in metric_dict.items():
        thisMetric = []
        for i in datasets:
            thisMetric.append(metFunc(i[target_col], clf.predict(i[feature_cols])))
        group_result[metricName] = thisMetric
    return group_result

In [25]:
run_results['metrics'] = get_metrics(mutated_data, classification_metrics)

In [26]:
np.arange(0,1.01, 0.1)

array([0. , 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1. ])

In [27]:
all_runs = []
for sev in np.arange(0,1.01, 0.1):
    sev = round(sev,2)
    mutated_data = get_mutations(datasets, sev)
    print(sev, mutated_data[0]['PageValues'].mean())
    run_results = {}
    run_results['mutation'] = 'change_to_zero'
    run_results['severity'] = sev
    run_results['metrics'] = get_metrics(mutated_data, classification_metrics)
    print((run_results['metrics']['f1']))
    all_runs.append(run_results)

0.0 0.0
[0.01098901098901099, 0.0, 0.01092896174863388, 0.0, 0.009345794392523364]
0.1 5.33193186625142
[0.48507462686567165, 0.4230769230769231, 0.45018450184501846, 0.4306569343065693, 0.4630225080385852]


C:\Users\chris\PycharmProjects\data-quality-ml-research\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


0.2 4.893666859026764
[0.4521072796934866, 0.3888888888888889, 0.4153846153846154, 0.3939393939393939, 0.4068965517241379]
0.3 4.150401347793188
[0.424, 0.30833333333333335, 0.373015873015873, 0.3607843137254902, 0.3701067615658363]
0.4 3.4895701104525543
[0.3598326359832636, 0.2672413793103448, 0.3292181069958848, 0.3426294820717131, 0.33210332103321033]
0.5 2.7873565491735603
[0.28193832599118945, 0.2290748898678414, 0.2832618025751073, 0.30327868852459017, 0.2943396226415094]
0.6 2.1846617755279807
[0.23853211009174313, 0.18518518518518517, 0.23529411764705882, 0.25316455696202533, 0.22924901185770752]
0.7 1.7978773210210868
[0.20657276995305165, 0.15165876777251186, 0.1895734597156398, 0.20087336244541484, 0.1589958158995816]
0.8 1.3806620094655313
[0.1642512077294686, 0.09950248756218906, 0.10050251256281408, 0.12206572769953052, 0.11255411255411256]
0.9 0.7692106840502839
[0.10204081632653061, 0.05154639175257732, 0.052083333333333336, 0.05025125628140704, 0.05357142857142857]
1.

C:\Users\chris\PycharmProjects\data-quality-ml-research\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [28]:
pd.DataFrame(all_runs)['metrics'].apply(lambda x: np.mean(x['precision']))

0     0.366667
1     0.723273
2     0.728746
3     0.720328
4     0.710861
5     0.682756
6     0.678530
7     0.663051
8     0.636208
9     0.608333
10    0.366667
Name: metrics, dtype: float64

In [29]:
pd.DataFrame(all_runs)

,mutation,severity,metrics
0,change_to_zero,0.0,"{'precision': [1.0, 0.0, 0.3333333333333333, 0..."
1,change_to_zero,0.1,"{'precision': [0.7471264367816092, 0.743243243..."
2,change_to_zero,0.2,"{'precision': [0.7375, 0.7424242424242424, 0.6..."
3,change_to_zero,0.3,"{'precision': [0.7681159420289855, 0.685185185..."
4,change_to_zero,0.4,"{'precision': [0.7413793103448276, 0.673913043..."
5,change_to_zero,0.5,"{'precision': [0.6956521739130435, 0.634146341..."
6,change_to_zero,0.6,"{'precision': [0.7027027027027027, 0.666666666..."
7,change_to_zero,0.7,"{'precision': [0.6875, 0.64, 0.645161290322580..."
8,change_to_zero,0.8,"{'precision': [0.6538461538461539, 0.666666666..."
9,change_to_zero,0.9,"{'precision': [0.6666666666666666, 0.625, 0.41..."


In [30]:
sorted(list(zip(feature_cols, clf.coef_[0])), key = lambda x: x[1])

[('OperatingSystems', -0.6634639529573585),
 ('SpecialDay', -0.11292733462403284),
 ('ExitRates', -0.06818158719744484),
 ('Region', -0.06362652158086866),
 ('BounceRates', -0.044281225365185556),
 ('Browser', -0.04275467033699612),
 ('TrafficType', -0.03890079803065126),
 ('Administrative_Duration', -0.0003287820631494156),
 ('Informational_Duration', -0.00022315530260401493),
 ('ProductRelated_Duration', 4.8452473277715564e-05),
 ('ProductRelated', 0.0036863215191721937),
 ('Administrative', 0.018438077809447416),
 ('Informational', 0.03570251463266292),
 ('PageValues', 0.0867820431917743)]

In [31]:
pd.DataFrame(all_runs)#.to_csv('../output_data/output.csv')

,mutation,severity,metrics
0,change_to_zero,0.0,"{'precision': [1.0, 0.0, 0.3333333333333333, 0..."
1,change_to_zero,0.1,"{'precision': [0.7471264367816092, 0.743243243..."
2,change_to_zero,0.2,"{'precision': [0.7375, 0.7424242424242424, 0.6..."
3,change_to_zero,0.3,"{'precision': [0.7681159420289855, 0.685185185..."
4,change_to_zero,0.4,"{'precision': [0.7413793103448276, 0.673913043..."
5,change_to_zero,0.5,"{'precision': [0.6956521739130435, 0.634146341..."
6,change_to_zero,0.6,"{'precision': [0.7027027027027027, 0.666666666..."
7,change_to_zero,0.7,"{'precision': [0.6875, 0.64, 0.645161290322580..."
8,change_to_zero,0.8,"{'precision': [0.6538461538461539, 0.666666666..."
9,change_to_zero,0.9,"{'precision': [0.6666666666666666, 0.625, 0.41..."


In [33]:
config

{'model': 'Logistic Regression',
 'data_location': '../data/online_shoppers_intention.csv',
 'preprocess_data': False,
 'mutator': 'change_to_zero',
 'num_scoring_groups': 5,
 'seeds': {'outer_seed': 123, 'inner_seed': 456}}

In [75]:
import time, json

In [65]:
run_timestamp = str(int(time.time()))
data_file_name = config['data_location'].split('/')[-1].split('.')[0]

In [66]:
from pathlib import Path

output_file = 'results.csv'
output_dir = Path('../output_data/' + '_'.join([data_file_name, run_timestamp]))

output_dir.mkdir(parents=True, exist_ok=True)

In [73]:
pd.DataFrame(all_runs).to_csv(output_dir / output_file)

In [78]:
with open(output_dir / 'config.json', 'w') as f:
    json.dump(config, f)